In [1]:
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

os.environ["MLFLOW_TRACKING_USERNAME"] = "giomamaca"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "095241f97fd50006565a872843e1054bb2ae82cb"
mlflow.set_tracking_uri('https://dagshub.com/giomamaca/ML2.mlflow')
# Find model version
client = MlflowClient()
for mv in client.search_model_versions("name='RandomForest_Fraud_Pipeline'"):
    print(f"Version: {mv.version}  |  Stage: {mv.current_stage}  |  Run ID: {mv.run_id}")
# Promote to Production
client.transition_model_version_stage(
    name='RandomForest_Fraud_Pipeline',
    version=1,  # ← change if your version number differs
    stage='Production'
)
# Load from Registry
pipeline = mlflow.sklearn.load_model('models:/RandomForest_Fraud_Pipeline/Production')
print('Pipeline loaded:', pipeline)
print('Steps:', [name for name, _ in pipeline.steps])
# Load raw test data
test_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

test = test_trans.merge(test_id, on='TransactionID', how='left')
print('Test shape:', test.shape)
# Predict
preds = pipeline.predict_proba(test)[:, 1]
print(f'Predictions shape : {preds.shape}')
print(f'Pred min/max/mean : {preds.min():.4f} / {preds.max():.4f} / {preds.mean():.4f}')
# Generate submission
submission = pd.DataFrame({
    'TransactionID': test['TransactionID'],
    'isFraud': preds
})
submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print(f'Submission shape: {submission.shape}')
print('Saved to submission.csv ✓')

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_22764\3112884941.py:19: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


RestException: RESOURCE_DOES_NOT_EXIST: Response: {'error_code': 'RESOURCE_DOES_NOT_EXIST'}